# New Models for Univarient Timeseries Forecast

## Imports and File Paths [randomstatsmodels](https://pypi.org/project/randomstatsmodels/)

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:

# ---- Setup: installs & imports ----
%pip -q install --upgrade randomstatsmodels pandas numpy matplotlib

import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.rcParams['figure.figsize'] = (10, 4)

# REQUIRED models from randomstatsmodels
try:
    from randomstatsmodels import AutoNEO, AutoFourier, AutoKNN, AutoPolymath, AutoThetaAR
except Exception as e:
    raise RuntimeError(
        "randomstatsmodels missing required classes. Please install a build that includes: "
        "AutoNEO, AutoFourier, AutoKNN, AutoPolymath, AutoThetaAR."
    ) from e

print("✅ Imports OK (randomstatsmodels AutoSuite)")


In [ ]:

# ---- Metrics & helpers ----
def mae(y, yhat):
    y, yhat = np.asarray(y), np.asarray(yhat)
    return float(np.mean(np.abs(y - yhat)))

def rmse(y, yhat):
    y, yhat = np.asarray(y), np.asarray(yhat)
    return float(np.sqrt(np.mean((y - yhat)**2)))

def mape(y, yhat):
    y, yhat = np.asarray(y), np.asarray(yhat)
    mask = y != 0
    return float(np.mean(np.abs((y[mask] - yhat[mask]) / y[mask])) * 100) if mask.any() else np.nan

def smape(y, yhat):
    y, yhat = np.asarray(y), np.asarray(yhat)
    denom = (np.abs(y) + np.abs(yhat)) / 2.0
    mask = denom != 0
    return float(np.mean(np.abs(y[mask] - yhat[mask]) / denom[mask]) * 100) if mask.any() else np.nan

def _as_series(preds, last_index, horizon):
    idx = pd.date_range(last_index + pd.Timedelta(days=1), periods=horizon, freq="D")
    return pd.Series(np.asarray(preds).ravel(), index=idx)

print("✅ Metrics ready")

In [ ]:

# ---- Config ----
TARGET_STORE = 1
TARGET_FAMILY = "GROCERY I"
HORIZON = 16                     # try 30 for month-ahead
ENSEMBLE_SCHEME = "inv_rmse"     # 'inv_rmse' or 'mean'

# Toggle models
USE_NEO = True
USE_FOURIER = True
USE_KNN = True
USE_POLYMATH = True
USE_THETAAR = True

print("✅ Config set")


In [ ]:

# ---- Load dataset & build series ----
CANDIDATES = [
    "/kaggle/input/store-sales-time-series-forecasting",
    "/kaggle/working",
    "../input/store-sales-time-series-forecasting",
    "./data",
    "."
]
DATA_DIR = next((c for c in CANDIDATES if os.path.exists(os.path.join(c, "train.csv"))), None) or "."
print("DATA_DIR =", DATA_DIR)

train_path = os.path.join(DATA_DIR, "train.csv")
if not os.path.exists(train_path):
    raise FileNotFoundError("Couldn't find train.csv. Set DATA_DIR to your dataset folder.")

df = pd.read_csv(train_path, parse_dates=["date"])

# Build daily target series for chosen store/family
y = (df.loc[(df["store_nbr"] == TARGET_STORE) & (df["family"] == TARGET_FAMILY), ["date", "sales"]]
        .sort_values("date")
        .set_index("date")
        .asfreq("D")["sales"]
        .fillna(0.0))

# Train/valid split
split_date = y.index.max() - pd.Timedelta(days=HORIZON-1)
y_tr = y.loc[y.index < split_date]
y_va = y.loc[y.index >= split_date]

print(f"Train: {y_tr.index.min().date()} → {y_tr.index.max().date()} (n={len(y_tr)})")
print(f"Valid: {y_va.index.min().date()} → {y_va.index.max().date()} (n={len(y_va)})")

ax = y.plot(title=f"Series Preview — Store {TARGET_STORE} | {TARGET_FAMILY}")
ax.set_ylabel("sales")
plt.show()

In [ ]:

# ---- Fit models & evaluate on validation ----
preds = {}

if USE_NEO:
    m = AutoNEO(); m.fit(y_tr)
    preds["AutoNEO"] = _as_series(m.predict(len(y_va)), y_tr.index[-1], len(y_va))

if USE_FOURIER:
    m = AutoFourier(); m.fit(y_tr)
    preds["AutoFourier"] = _as_series(m.predict(len(y_va)), y_tr.index[-1], len(y_va))

if USE_KNN:
    m = AutoKNN(); m.fit(y_tr)
    preds["AutoKNN"] = _as_series(m.predict(len(y_va)), y_tr.index[-1], len(y_va))

if USE_POLYMATH:
    m = AutoPolymath(); m.fit(y_tr)
    preds["AutoPolymath"] = _as_series(m.predict(len(y_va)), y_tr.index[-1], len(y_va))

if USE_THETAAR:
    m = AutoThetaAR(); m.fit(y_tr)
    preds["AutoThetaAR"] = _as_series(m.predict(len(y_va)), y_tr.index[-1], len(y_va))

rows = [{
    "Model": name,
    "MAE": mae(y_va, fc),
    "RMSE": rmse(y_va, fc),
    "MAPE": mape(y_va, fc),
    "sMAPE": smape(y_va, fc)
} for name, fc in preds.items()]

eval_df = pd.DataFrame(rows).sort_values("RMSE")
print("Validation metrics (sorted by RMSE):")
print(eval_df.to_string(index=False))

# Plot validation forecasts
ax = y_tr[-120:].plot(label="train")
y_va.plot(ax=ax, label="valid", alpha=0.7)
for name, fc in preds.items():
    fc.plot(ax=ax, label=name)
ax.set_title("Validation forecasts")
ax.legend(); plt.show()

In [ ]:

# ---- Ensemble weights & final forecast ----
weights = {}
if ENSEMBLE_SCHEME.lower() == "inv_rmse":
    rm = {k: rmse(y_va, v) for k, v in preds.items() if np.isfinite(rmse(y_va, v)) and rmse(y_va, v) > 0}
    if rm:
        inv = {k: 1.0/v for k, v in rm.items()}
        s = sum(inv.values())
        weights = {k: w/s for k, w in inv.items()}
        print("Inverse-RMSE weights:", weights)
    else:
        print("No valid RMSEs; falling back to equal average.")
else:
    print("Using simple mean ensemble.")

components = {}
if USE_NEO:
    m = AutoNEO(); m.fit(y)
    components["AutoNEO"] = _as_series(m.predict(HORIZON), y.index[-1], HORIZON)
if USE_FOURIER:
    m = AutoFourier(); m.fit(y)
    components["AutoFourier"] = _as_series(m.predict(HORIZON), y.index[-1], HORIZON)
if USE_KNN:
    m = AutoKNN(); m.fit(y)
    components["AutoKNN"] = _as_series(m.predict(HORIZON), y.index[-1], HORIZON)
if USE_POLYMATH:
    m = AutoPolymath(); m.fit(y)
    components["AutoPolymath"] = _as_series(m.predict(HORIZON), y.index[-1], HORIZON)
if USE_THETAAR:
    m = AutoThetaAR(); m.fit(y)
    components["AutoThetaAR"] = _as_series(m.predict(HORIZON), y.index[-1], HORIZON)

if not components:
    raise RuntimeError("No models enabled; enable at least one.")

if weights and len(components) > 1:
    idx = next(iter(components.values())).index
    final = sum(components[k].reindex(idx).fillna(method="ffill") * w for k, w in weights.items() if k in components)
else:
    final = pd.concat(components.values(), axis=1).mean(axis=1) if len(components) > 1 else next(iter(components.values()))

out_path = "../AutoSuite_forecast_FINAL_store{}_{}.csv".format(TARGET_STORE, TARGET_FAMILY.replace(" ", "_"))
pd.DataFrame({"date": final.index, "store_nbr": TARGET_STORE, "family": TARGET_FAMILY, "sales": final.values}).to_csv(out_path, index=False)
print("📄 Saved forecast CSV:", out_path)

ax = y[-120:].plot(label="history")
final.plot(ax=ax, label="forecast")
ax.set_title("Final forecast (next HORIZON days)")
ax.legend(); plt.show()